In [ ]:
import torch
import torchvision
import torchvision.transforms.v2
from torch.utils.data import DataLoader

transforms = torchvision.transforms.v2.Compose([
    torchvision.transforms.v2.Resize(224),
    torchvision.transforms.v2.Grayscale(num_output_channels=3),
    torchvision.transforms.v2.ToImage(),
    torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
    torchvision.transforms.v2.Normalize((0.1307,), (0.3081,)),
])
train_ds = torchvision.datasets.MNIST(
    "mnist", train=True, download=True, transform=transforms
)
test_ds = torchvision.datasets.MNIST(
    "mnist", train=False, download=True, transform=transforms
)

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=128, shuffle=False)

In [20]:
torchvision.models.vit_b_16(num_classes = 10)(next(iter(train_dl))[0]).shape

RuntimeError: Given groups=1, weight of size [768, 3, 16, 16], expected input[128, 1, 224, 224] to have 3 channels, but got 1 channels instead

In [9]:
import numpy
import scipy.special
import sklearn.metrics

def estimate_quality(
    y_pred_logits: numpy.ndarray, y_true: numpy.ndarray
) -> dict:
    y_pred_proba = scipy.special.softmax(y_pred_logits, axis=1)
    y_pred = numpy.argmax(y_pred_proba, axis=1)
    return {
        "Accuracy": sklearn.metrics.accuracy_score(y_true, y_pred),
        "AUC-ROC": sklearn.metrics.roc_auc_score(
            y_true, y_pred_proba, multi_class="ovo"
        ),
        "Precision": sklearn.metrics.precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Recall": sklearn.metrics.recall_score(
            y_true, y_pred, average="macro"
        ),
        "F1-score": sklearn.metrics.f1_score(
            y_true, y_pred, average="macro"
        ),
        "TOP-5 Accuracy": sklearn.metrics.top_k_accuracy_score(
            y_true, y_pred_proba, k=5
        )
    }

def set_random_seed(seed: int = 42):
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True

In [10]:
import tqdm
import torchvision.models

device = torch.device('cuda')

def train_eval(epochs: int = 5):
    set_random_seed(42)
    model = torchvision.models.resnet18(num_classes = 10)
    model.conv1 = torch.nn.Conv2d(in_channels=1, out_channels=64, kernel_size=7, stride=2, padding=3)
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters())

    pbar = tqdm.trange(epochs)
    history = []
    for epoch in pbar:
        model.train()
        for X, y in train_dl:
            optimizer.zero_grad()
            pred = model(X.to(device))
            loss = torch.nn.functional.cross_entropy(pred, y.to(device))
            loss.backward()
            optimizer.step()

        model.eval()
        test_preds, test_targets = [], []
        for X, y in test_dl:
            test_targets.extend(y.tolist())
            with torch.no_grad():
                test_preds.extend(model(X.to(device)).cpu().tolist())
        metrics = estimate_quality(
            numpy.array(test_preds), numpy.array(test_targets)
        )
        pbar.set_postfix(metrics)
        history.append(metrics)
    return history

In [11]:
train_eval()

100%|██████████| 5/5 [01:54<00:00, 22.98s/it, Accuracy=0.99, AUC-ROC=1, Precision=0.99, Recall=0.99, F1-score=0.99, TOP-5 Accuracy=1]    


[{'Accuracy': 0.9768,
  'AUC-ROC': 0.9995802100045356,
  'Precision': 0.9766169816560867,
  'Recall': 0.9768842599924928,
  'F1-score': 0.9766619839793342,
  'TOP-5 Accuracy': 1.0},
 {'Accuracy': 0.9854,
  'AUC-ROC': 0.9998189881691116,
  'Precision': 0.9853544708172357,
  'Recall': 0.9851914296748949,
  'F1-score': 0.9851942209936706,
  'TOP-5 Accuracy': 0.9999},
 {'Accuracy': 0.9886,
  'AUC-ROC': 0.9998933689715837,
  'Precision': 0.9886886883041427,
  'Recall': 0.9885633038658668,
  'F1-score': 0.9885798426297988,
  'TOP-5 Accuracy': 0.9999},
 {'Accuracy': 0.9918,
  'AUC-ROC': 0.9999339072903157,
  'Precision': 0.991797885903301,
  'Recall': 0.9917616500326039,
  'F1-score': 0.991768031318385,
  'TOP-5 Accuracy': 0.9998},
 {'Accuracy': 0.99,
  'AUC-ROC': 0.9999207797367259,
  'Precision': 0.9898678870129485,
  'Recall': 0.9898593530674706,
  'F1-score': 0.9898288238928835,
  'TOP-5 Accuracy': 0.9999}]